In [1]:
# =========================
# Load data and show head
# =========================

import os
import random
import pandas as pd

TRAIN_CSV = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

# Optional: if you ever attach a separate tokenizer artifact dataset, set it here.
# Leave as None to use the tokenizer from MODEL_PATH.
LOCAL_TOKENIZER_PATH = None

OUTPUT_DIR = "/kaggle/working/adapter_output"

RANDOM_SEED = 42
TRAIN_ROWS_LIMIT = 2000   # raise later if training is stable
VAL_FRACTION = 0.05

random.seed(RANDOM_SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(TRAIN_CSV)
print(df.head())

if TRAIN_ROWS_LIMIT is not None:
    df = df.iloc[:TRAIN_ROWS_LIMIT].copy()

df = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

val_size = max(1, int(len(df) * VAL_FRACTION))
val_df = df.iloc[:val_size].copy()
train_df = df.iloc[val_size:].copy()

print(f"train={len(train_df)} val={len(val_df)}")


import sys
import os
import shutil
import stat

# ptxas-blackwell fix
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"]           = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst

    try:
        import triton.backends.nvidia.compiler as nv_compiler
        def _safe_get_ptxas_version(*args, **kwargs):
            return (12, 9, 0)
        nv_compiler.get_ptxas_version = _safe_get_ptxas_version
        print("Triton ptxas fix applied ✓")
    except Exception as e:
        print(f"Triton ptxas partial fix (non-fatal): {e}")
else:
    print("ptxas-blackwell not found — continuing without fix")

         id                                             prompt  \
0  00066667  In Alice's Wonderland, a secret bit manipulati...   
1  000b53cf  In Alice's Wonderland, a secret bit manipulati...   
2  00189f6a  In Alice's Wonderland, secret encryption rules...   
3  001b24c4  In Alice's Wonderland, numbers are secretly co...   
4  001c63cb  In Alice's Wonderland, secret encryption rules...   

                  answer  
0               10010111  
1               01000011  
2      cat imagines book  
3                XXXVIII  
4  wizard creates secret  
train=1900 val=100


In [2]:
# =========================
# Create adapter (offline RTX Pro 6000 version)
# =========================

import os
import site
import importlib.util
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

# Make local NVIDIA utility packages visible
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script")
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU is required.")

print("gpu:", torch.cuda.get_device_name(0))
print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
print("cutlass spec:", importlib.util.find_spec("cutlass"))
print("mamba_ssm spec:", importlib.util.find_spec("mamba_ssm"))
print("sentencepiece spec:", importlib.util.find_spec("sentencepiece"))

# Fail fast if the local custom-code dependencies are not visible
if importlib.util.find_spec("cutlass") is None:
    raise RuntimeError(
        "Local CUTLASS Python package is not visible. "
        "The Kaggle NVIDIA utility path was added, but 'cutlass' still was not found."
    )

if importlib.util.find_spec("mamba_ssm") is None:
    raise RuntimeError(
        "Local mamba_ssm package is not visible. "
        "The Kaggle NVIDIA utility path was added, but 'mamba_ssm' still was not found."
    )

# ---- config ----
SYSTEM_PROMPT = (
    "You are a careful mathematical reasoning assistant. "
    "For each test case, generate a response and place the final answer "
    "inside a single LaTeX \\boxed{} command."
)

MAX_PROMPT_TOKENS = 256
MAX_ANSWER_TOKENS = 64
MAX_TOTAL_TOKENS = 384

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 1
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.03
LOGGING_STEPS = 10

PROMPT_TEMPLATE = (
    "Solve the following problem. Put the final answer inside \\boxed{{}}.\n\n"
    "Problem:\n{question}"
)

def load_tokenizer_offline(model_path: str, tokenizer_fallback_path: str | None = None):
    candidates = []
    if tokenizer_fallback_path and os.path.exists(tokenizer_fallback_path):
        candidates.append(tokenizer_fallback_path)
    candidates.append(model_path)

    errors = []

    for candidate in candidates:
        # Try fast tokenizer first; this avoids sentencepiece if tokenizer.json exists
        try:
            tok = AutoTokenizer.from_pretrained(
                candidate,
                trust_remote_code=True,
                use_fast=True,
                local_files_only=True,
            )
            print(f"Loaded FAST tokenizer from: {candidate}")
            return tok
        except Exception as e:
            errors.append(f"FAST tokenizer failed at {candidate}: {repr(e)}")

        # Fall back to slow tokenizer only if sentencepiece exists
        if importlib.util.find_spec("sentencepiece") is not None:
            try:
                tok = AutoTokenizer.from_pretrained(
                    candidate,
                    trust_remote_code=True,
                    use_fast=False,
                    local_files_only=True,
                )
                print(f"Loaded SLOW tokenizer from: {candidate}")
                return tok
            except Exception as e:
                errors.append(f"SLOW tokenizer failed at {candidate}: {repr(e)}")

    msg = "\n".join(errors)
    raise RuntimeError(
        "Offline tokenizer loading failed.\n"
        "Because this RTX session has no internet, do not try pip installs here.\n"
        "Either:\n"
        "1) make sure the tokenizer files inside MODEL_PATH are sufficient for fast loading, or\n"
        "2) attach a separate tokenizer artifact dataset and point LOCAL_TOKENIZER_PATH to it.\n\n"
        f"Details:\n{msg}"
    )

tokenizer = load_tokenizer_offline(MODEL_PATH, LOCAL_TOKENIZER_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def choose_answer_variant(answer: str) -> str:
    answer = str(answer).strip()
    if "\\boxed{" in answer:
        return answer
    return f"The final answer is \\boxed{{{answer}}}."

class BoxedReasoningDataset(Dataset):
    def __init__(self, df_in, tokenizer):
        self.df = df_in.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        prompt = PROMPT_TEMPLATE.format(question=row["prompt"])
        answer = choose_answer_variant(row["answer"])

        prompt_text = (
            f"<|system|>\n{SYSTEM_PROMPT}\n"
            f"<|user|>\n{prompt}\n"
            f"<|assistant|>\n"
        )

        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_PROMPT_TOKENS,
        )["input_ids"]

        answer_ids = self.tokenizer(
            answer,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_ANSWER_TOKENS,
        )["input_ids"]

        eos_id = self.tokenizer.eos_token_id
        input_ids = prompt_ids + answer_ids + ([eos_id] if eos_id is not None else [])
        labels = [-100] * len(prompt_ids) + answer_ids + ([eos_id] if eos_id is not None else [])

        input_ids = input_ids[:MAX_TOTAL_TOKENS]
        labels = labels[:MAX_TOTAL_TOKENS]
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

class DynamicPadCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)

        def pad_tensor(t, pad_value):
            if len(t) == max_len:
                return t
            pad = torch.full((max_len - len(t),), pad_value, dtype=t.dtype)
            return torch.cat([t, pad], dim=0)

        return {
            "input_ids": torch.stack([pad_tensor(x["input_ids"], self.pad_token_id) for x in features]),
            "attention_mask": torch.stack([pad_tensor(x["attention_mask"], 0) for x in features]),
            "labels": torch.stack([pad_tensor(x["labels"], -100) for x in features]),
        }

# Load full model directly in BF16 on the RTX Pro 6000.
# No bitsandbytes, no QLoRA, no internet install dependency.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    local_files_only=True,
    device_map={"": 0},   # place the full model on GPU 0
)

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_dataset = BoxedReasoningDataset(train_df, tokenizer)
val_dataset = BoxedReasoningDataset(val_df, tokenizer)
collator = DynamicPadCollator(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    eval_strategy="no",
    bf16=True,
    fp16=False,
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)
sample = train_dataset[0]
print("sample lengths:", {k: v.shape for k, v in sample.items()})
print(tokenizer.decode(sample["input_ids"][:200], skip_special_tokens=False))
trainer.train()

# Save adapter files, including adapter_config.json
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved files:", os.listdir(OUTPUT_DIR))
print("adapter_config.json exists:", os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json")))

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


torch: 2.12.0.dev20260324+cu128
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
vram_gb: 94.97
cutlass spec: ModuleSpec(name='cutlass', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7bb2a9342b10>, origin='/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/cutlass/__init__.py', submodule_search_locations=['/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/cutlass'])
mamba_ssm spec: ModuleSpec(name='mamba_ssm', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7bb2a843ac90>, origin='/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/mamba_ssm/__init__.py', submodule_search_locations=['/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/mamba_ssm'])
sentencepiece spec: ModuleSpec(name='sentencepiece', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7bb52f8c92e0>, origin='/usr/local/lib/python3.12/dist-packages/

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 220,034,560 || all params: 31,797,971,904 || trainable%: 0.6920
sample lengths: {'input_ids': torch.Size([274]), 'attention_mask': torch.Size([274]), 'labels': torch.Size([274])}
<|system|>
You are a careful mathematical reasoning assistant. For each test case, generate a response and place the final answer inside a single LaTeX \boxed{} command.
<|user|>
Solve the following problem. Put the final answer inside \boxed{}.

Problem:
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
00110111 -> 10001011
10101001 -> 01000100
01011010 -> 00001101
10001000 -> 01000100
01011100 -> 


PermissionError: [Errno 13] Permission denied: '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell'

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')